In [639]:
import pandas as pd
import altair as alt
alt.data_transformers.enable("vegafusion")


DataTransformerRegistry.enable('vegafusion')

In [640]:
data_q3 = pd.read_csv('../../data/data_Q3.csv')
data_q4 = pd.read_csv('../../data/data_Q4.csv')

In [641]:
data_q3

,season,number_in_season,episode_id,character,word_count,image
0,1,1,1,Barney,116.0,https://img.icons8.com/doodle/48/barney-gumble...
1,1,1,1,Bart,337.0,https://img.icons8.com/doodle/48/bart-simpson.png
2,1,1,1,Flanders,103.0,https://img.icons8.com/doodle/48/ned-flanders.png
3,1,1,1,Grampa,4.0,https://img.icons8.com/doodle/48/abraham-simps...
4,1,1,1,Homer,909.0,https://img.icons8.com/doodle/48/homer-simpson...
...,...,...,...,...,...,...
8583,26,16,568,Lisa,94.0,https://img.icons8.com/doodle/48/lisa-simpson.png
8584,26,16,568,Lou,18.0,https://static.simpsonswiki.com/images/c/cf/Ta...
8585,26,16,568,Marge,450.0,https://img.icons8.com/doodle/48/marge-simpson...
8586,26,16,568,Rev. Lovejoy,109.0,https://static.simpsonswiki.com/images/4/40/Ta...


In [642]:
characters= ['Lenny', 'Flanders']
season = 14
data_q3 = data_q3[data_q3['character'].isin(characters)]
data_q3 = data_q3[data_q3['season'] == season]
# Handle cases where one character doesn't speak by finding the character with max words per episode
data_q3['more_words'] = data_q3.groupby(['season', 'number_in_season'], group_keys=False).apply(
    lambda x: pd.Series([x['character'].iloc[x['word_count'].argmax()]] * len(x), index=x.index),
    include_groups=False
)
data_q3

,season,number_in_season,episode_id,character,word_count,image,more_words
4316,14,1,292,Flanders,111.0,https://img.icons8.com/doodle/48/ned-flanders.png,Flanders
4319,14,1,292,Lenny,7.0,https://static.simpsonswiki.com/images/6/6f/Ta...,Flanders
4335,14,2,293,Lenny,81.0,https://static.simpsonswiki.com/images/6/6f/Ta...,Lenny
4342,14,3,294,Flanders,50.0,https://img.icons8.com/doodle/48/ned-flanders.png,Flanders
4360,14,4,295,Lenny,7.0,https://static.simpsonswiki.com/images/6/6f/Ta...,Lenny
4378,14,5,296,Flanders,16.0,https://img.icons8.com/doodle/48/ned-flanders.png,Lenny
4382,14,5,296,Lenny,35.0,https://static.simpsonswiki.com/images/6/6f/Ta...,Lenny
4399,14,6,297,Lenny,24.0,https://static.simpsonswiki.com/images/6/6f/Ta...,Lenny
4411,14,7,298,Flanders,14.0,https://img.icons8.com/doodle/48/ned-flanders.png,Flanders
4426,14,8,299,Flanders,34.0,https://img.icons8.com/doodle/48/ned-flanders.png,Flanders


In [643]:
data_q3.columns

Index(['season', 'number_in_season', 'episode_id', 'character', 'word_count',
       'image', 'more_words'],
      dtype='object')

In [644]:
selector = alt.selection_point(fields=['episode_id'])


chart = (
    alt.Chart(data_q3)
    ).encode(
        x=alt.X("word_count:Q", title="Word Count"),
        y=alt.Y("number_in_season:O", title="Episode Number in Season"),
        detail="number_in_season:O",
        tooltip=["season:O", "number_in_season:O", "character:N", "word_count:Q"],
        color=alt.Color('more_words:N', title='More Words', scale=alt.Scale(domain=[characters[0], characters[1]], range=['#0e33eb', '#f40c0c'])),
        opacity=alt.condition(selector, alt.value(1), alt.value(0.2))
    )



line = chart.mark_line(strokeWidth=4)

faces = chart.mark_image(width=20, height=20).encode(
    url="image:N",

)

points = chart.mark_point(size=500, filled=True).encode(
    color = alt.Color('character:N', title='Character', scale=alt.Scale(domain=[characters[0], characters[1]], range=['#0e33eb', '#f40c0c']))
)

q3 =(line + points + faces).properties(
    title=f"{characters[0]} vs {characters[1]} Word Count by Episode (Season {season})",
    width=600,
    height=600
).add_params(selector)
q3

alt.LayerChart(...)

In [645]:
data_q4 = data_q4[data_q4['season'] == season]

In [646]:
base = alt.Chart(data_q4).transform_window(
    cumulative_words='sum(word_count)',
    sort=[alt.SortField('number_in_season')],
    groupby=['character']
).encode(
    x=alt.X('number_in_season:Q', scale = alt.Scale(domain=[0, data_q4['number_in_season'].max()]), title='Episode Number in Season'),
    y=alt.Y('cumulative_words:Q'),
    color =alt.Color('character:N', legend=None),
    tooltip=[
        alt.Tooltip('character:N'),
        alt.Tooltip('number_in_season:Q'),
        alt.Tooltip('cumulative_words:Q', title='Total Words Spoken')
    ]
)

highlight_lines = base.mark_line(strokeWidth=2, point=True).transform_filter(
    alt.FieldOneOfPredicate(field='character', oneOf=[characters[0], characters[1]])
)

faces = base.mark_image(width=40, height=40).transform_calculate(
    number_in_season_1 = f"{data_q4['number_in_season'].max() + 1}"
).encode(
    x=alt.X('number_in_season_1:Q'),
    url='image:N'
).transform_filter(
    alt.FieldOneOfPredicate(field='character', oneOf=[characters[0], characters[1]])
).transform_window(
    rank='rank()',
    sort=[alt.SortField('number_in_season', order='descending')],
    groupby=['character']
).transform_filter(
    alt.datum.rank == 1)


q4_all = ( highlight_lines + faces).properties(
    width=600, height=600, title=f'Cumulative Word Count All season {season} Episodes'
)

q4_all


alt.LayerChart(...)

In [647]:
data_q4 = data_q4[data_q4['season'] == season]
# episode_id = 293
# data_q4 = data_q4[data_q4['episode_id'] == episode_id]
data_q4.columns

Index(['season', 'number_in_season', 'episode_id', 'line_number_in_episode',
       'character', 'word_count', 'image'],
      dtype='object')

In [648]:
data_q4

,season,number_in_season,episode_id,line_number_in_episode,character,word_count,image
46661,14,2,293,184,Lenny,2.0,https://static.simpsonswiki.com/images/6/6f/Ta...
46955,14,1,292,1,Flanders,17.0,https://img.icons8.com/doodle/48/ned-flanders.png
46956,14,1,292,2,Marge,11.0,https://img.icons8.com/doodle/48/marge-simpson...
46957,14,1,292,3,Flanders,6.0,https://img.icons8.com/doodle/48/ned-flanders.png
46958,14,1,292,4,Marge,9.0,https://img.icons8.com/doodle/48/marge-simpson...
...,...,...,...,...,...,...,...
91639,14,17,308,281,Marge,41.0,https://img.icons8.com/doodle/48/marge-simpson...
92016,14,10,301,118,Homer,22.0,https://img.icons8.com/doodle/48/homer-simpson...
92061,14,4,295,230,Marge,8.0,https://img.icons8.com/doodle/48/marge-simpson...
92062,14,17,308,246,Marge,6.0,https://img.icons8.com/doodle/48/marge-simpson...


In [666]:
import altair as alt

# Shared scales
x_scale = alt.Scale(zero=False, nice=False, padding=0)
y_scale = alt.Scale(zero=True)

# Base transformed data
base = (
    alt.Chart(data_q4)
    .transform_filter(selector)
    .transform_window(
        cumulative_words='sum(word_count)',
        sort=[alt.SortField('line_number_in_episode')],
        groupby=['character']
    )
    .transform_joinaggregate(
        max_line_number='max(line_number_in_episode)'
    )
)

domain_anchor = (
    alt.Chart(data_q4)
    .transform_filter(selector)
    .transform_joinaggregate(
        max_line_number='max(line_number_in_episode)'
    )
    .transform_calculate(
        x0='0',
        x1='datum.max_line_number'
    )
    .mark_rule(opacity=0)
    .encode(
        x=alt.X('x0:Q', scale=x_scale),
        x2='x1:Q'
    )
)

highlight_lines = (
    base
    .mark_line(strokeWidth=2, point=True)
    .transform_filter(
        alt.FieldOneOfPredicate(
            field='character',
            oneOf=[characters[0], characters[1]]
        )
    )
    .encode(
        x=alt.X(
            'line_number_in_episode:Q',
            title='Line Number in Episode',
            scale=x_scale
        ),
        y=alt.Y(
            'cumulative_words:Q',
            title='Cumulative Words Spoken',
            scale=y_scale
        ),
        color=alt.Color('character:N', legend=None),
        tooltip=[
            alt.Tooltip('character:N'),
            alt.Tooltip('line_number_in_episode:Q'),
            alt.Tooltip('cumulative_words:Q', title='Total Words Spoken')
        ]
    )
)

# Character face images outside the plot area
faces = (
    base
    .transform_filter(
        alt.FieldOneOfPredicate(
            field='character',
            oneOf=[characters[0], characters[1]]
        )
    )
    .transform_window(
        rank='rank()',
        sort=[alt.SortField('line_number_in_episode', order='descending')],
        groupby=['character']
    )
    .transform_filter(alt.datum.rank == 1)
    .mark_image(width=30, height=30, clip=False)
    .encode(
        x=alt.value(620),   # chart width (600) + 20px outside plot box
        y=alt.Y('cumulative_words:Q', scale=y_scale),
        url='image:N'
    )
)

# Final layered chart
q4_episode = (
    (domain_anchor + highlight_lines + faces)
    .properties(
        width=600,
        height=600,
        title="Cumulative Word Count"
    )
)

In [667]:
final_chart = alt.hconcat(q3,q4_episode)
final_chart

alt.HConcatChart(...)